# 🌲 California Housing — Random Forest Regression (Revised)

This notebook trains, evaluates, and interprets a Random Forest model for predicting median house values in California.

### **Workflow**
1. Load dataset
2. Feature engineering
3. Train / validation / test split
4. Preprocessing pipeline
5. Cross-validation (mean + std)
6. Hyperparameter tuning (RandomizedSearchCV)
7. Train final model
8. Train vs validation metrics (overfitting check)
9. Validation performance
10. Residual diagnostics
11. Feature importance (Permutation + RF)
12. Final test set evaluation
13. Save trained model

> **Revision notes:** Added feature engineering, hyperparameter tuning, train vs val comparison,
> CV std reporting, fixed deprecated `squared=False`, and added final test set evaluation.

<a href="https://colab.research.google.com/github/Robbie669/data-science-portfolio/blob/main/projects/california-housing/02_random_forest.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Run in Colab"/>
</a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.inspection import permutation_importance
from scipy.stats import randint

import joblib

sns.set(style="whitegrid", context="notebook")

## 1. Load Dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.head()

## 2. Feature Engineering

**[ADDED — minor]** Three interaction features that are well-known to improve RF performance on this dataset.
Tree-based models can discover these relationships on their own in theory, but providing them
explicitly reduces the depth needed and speeds up training.

In [ ]:
df["rooms_per_household"]    = df["AveRooms"]    / df["AveOccup"]
df["bedrooms_per_room"]      = df["AveBedrms"]   / df["AveRooms"]
df["population_per_household"] = df["Population"] / df["AveOccup"]

df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe()

## 3. Train / Validation / Test Split
A clean 3-way split prevents leakage and gives us an untouched hold-out for final evaluation.

In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 4. Preprocessing Pipeline
Random Forests don't require scaling, but it's included so the pipeline
can be swapped to a linear or SVM model without changes downstream.

In [ ]:
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features)
    ]
)

rf_base = RandomForestRegressor(n_jobs=-1, random_state=42)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf_base)
])

## 5. Cross-Validation (mean + std)

**[FIXED — critical]** Now reporting both mean and std of CV RMSE.
The std tells us how stable the estimate is across folds — a high std means the model is
sensitive to which data it sees.

In [ ]:
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

cv_rmse_mean = -cv_scores.mean()
cv_rmse_std  = cv_scores.std()

print(f"CV RMSE: {cv_rmse_mean:.4f} ± {cv_rmse_std:.4f}")

## 6. Hyperparameter Tuning

**[ADDED — critical]** The original notebook used default hyperparameters with `max_depth=None`,
meaning fully grown trees — a known overfitting risk.

We use `RandomizedSearchCV` over the key parameters that matter most for RF:
- `n_estimators`: more trees = lower variance, diminishing returns after ~200
- `max_depth`: controls tree depth; `None` = overfit, too shallow = underfit
- `min_samples_leaf`: minimum samples at a leaf; higher = more regularisation
- `max_features`: fraction of features considered per split

In [ ]:
param_dist = {
    "model__n_estimators":    randint(100, 500),
    "model__max_depth":       [None, 10, 20, 30, 40],
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features":    ["sqrt", "log2", 0.5, 0.8],
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print(f"Best CV RMSE: {-search.best_score_:.4f}")

## 7. Train Final Model

We use the best estimator found by the search, already fit on `X_train`.

In [ ]:
best_pipeline = search.best_estimator_
print("Final pipeline ready.")

## 8. Train vs Validation Metrics — Overfitting Check

**[ADDED — critical]** Without comparing train vs validation error, we cannot detect overfitting.
A large gap (low train RMSE, high val RMSE) signals the model has memorised the training data.

In [ ]:
train_preds = best_pipeline.predict(X_train)
val_preds   = best_pipeline.predict(X_val)

train_rmse = root_mean_squared_error(y_train, train_preds)
val_rmse   = root_mean_squared_error(y_val,   val_preds)

print(f"Train RMSE : {train_rmse:.4f}")
print(f"Val   RMSE : {val_rmse:.4f}")
print(f"Gap        : {val_rmse - train_rmse:.4f}  {'(acceptable)' if val_rmse - train_rmse < 0.05 else '(potential overfit — check)'} ")

## 9. Validation Performance

In [ ]:
rmse = root_mean_squared_error(y_val, val_preds)  # FIXED: was mean_squared_error(..., squared=False)
mae  = mean_absolute_error(y_val, val_preds)
r2   = r2_score(y_val, val_preds)

print(f"Val RMSE : {rmse:.4f}")
print(f"Val MAE  : {mae:.4f}")
print(f"Val R²   : {r2:.4f}")

## 10. Residual Diagnostics

In [ ]:
residuals = y_val - val_preds

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(val_preds, residuals, alpha=0.3)
plt.axhline(0, color="red")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residuals vs Predictions")
plt.show()

## 11. Feature Importance (Permutation)
Permutation importance is more reliable than built-in RF importance for high-cardinality features.

In [ ]:
result = permutation_importance(
    best_pipeline, X_val, y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importances = pd.DataFrame({
    "feature":    X.columns,
    "importance": result.importances_mean,
    "std":        result.importances_std
}).sort_values("importance", ascending=False)

importances

## 12. Final Test Set Evaluation

**[ADDED — critical]** The original notebook created `X_test` / `y_test` but never used them.
This is the true hold-out evaluation — only run once, on the best model.

In [ ]:
test_preds = best_pipeline.predict(X_test)

test_rmse = root_mean_squared_error(y_test, test_preds)
test_mae  = mean_absolute_error(y_test, test_preds)
test_r2   = r2_score(y_test, test_preds)

print("===== FINAL TEST SET RESULTS =====")
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R²   : {test_r2:.4f}")
print("=================================")

summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "RMSE":  [train_rmse, val_rmse, test_rmse],
    "MAE":   [mean_absolute_error(y_train, train_preds), mae, test_mae],
    "R2":    [r2_score(y_train, train_preds), r2, test_r2]
})
summary

## 13. Save Model

In [ ]:
joblib.dump(best_pipeline, "random_forest_california_housing_v2.pkl")
print("Model saved: random_forest_california_housing_v2.pkl")

## Summary of Changes from v1

| # | Category | Change |
|---|----------|--------|
| 1 | Critical | Added final test set evaluation cell (was never run in v1) |
| 2 | Critical | Added `RandomizedSearchCV` for hyperparameter tuning |
| 3 | Critical | Added train vs val metric comparison to detect overfitting |
| 4 | Critical | Fixed deprecated `mean_squared_error(..., squared=False)` → `root_mean_squared_error` |
| 5 | Minor    | Added feature engineering: `rooms_per_household`, `bedrooms_per_room`, `population_per_household` |
| 6 | Minor    | Added CV std deviation reporting (`±`) next to mean RMSE |